### Lab Assignment 2 – Multinomial Naive Bayes on `spam.csv`

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1')
df = df.drop(df.iloc[:, 2:], axis=1)
df.columns = ['Labels', 'SMS']

print(df['Labels'].value_counts())
df.head()

In [ ]:
new_labels = {
    'spam': 1,
    'ham': 0
}

df['Labels'] = df['Labels'].map(new_labels)

X = df['SMS'].to_numpy(dtype=object)
y = df['Labels'].to_numpy()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

### Part 1: CountVectorizer with stop words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

bow = CountVectorizer(stop_words='english')

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print(f'Number of features: {len(bow.get_feature_names_out())}')
print(f'Data dimensions: {X_train_bow.shape}')

In [ ]:
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

y_pred_train_bow = mnb_bow.predict(X_train_bow)
y_pred_test_bow = mnb_bow.predict(X_test_bow)

acc_train_bow = accuracy_score(y_train, y_pred_train_bow)
acc_test_bow = accuracy_score(y_test, y_pred_test_bow)

print(f'Training data accuracy: {acc_train_bow}')
print(f'Test data accuracy: {acc_test_bow}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_test_bow))
print('Classification Report:\n', classification_report(y_test, y_pred_test_bow))

### Part 2: TF-IDF with stop words

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Number of features: {len(tfidf.get_feature_names_out())}')
print(f'Data dimensions: {X_train_tfidf.shape}')

In [ ]:
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

y_pred_train_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_pred_test_tfidf = mnb_tfidf.predict(X_test_tfidf)

acc_train_tfidf = accuracy_score(y_train, y_pred_train_tfidf)
acc_test_tfidf = accuracy_score(y_test, y_pred_test_tfidf)

print(f'Training data accuracy: {acc_train_tfidf}')
print(f'Test data accuracy: {acc_test_tfidf}')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_test_tfidf))
print('Classification Report:\n', classification_report(y_test, y_pred_test_tfidf))

### Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    'Feature': ['CountVectorizer (stop words)', 'TF-IDF (stop words)'],
    'Train Accuracy': [acc_train_bow, acc_train_tfidf],
    'Test Accuracy': [acc_test_bow, acc_test_tfidf],
    'Spam Precision': [precision_score(y_test, y_pred_test_bow), precision_score(y_test, y_pred_test_tfidf)],
    'Spam Recall': [recall_score(y_test, y_pred_test_bow), recall_score(y_test, y_pred_test_tfidf)],
    'Spam F1': [f1_score(y_test, y_pred_test_bow), f1_score(y_test, y_pred_test_tfidf)]
})
results

**Conclusion:** Both models use English stop words. CountVectorizer reached about 0.983 test accuracy with spam recall of 0.90 and F1 of 0.94, while TF-IDF reached about 0.961 test accuracy with spam recall of only 0.73 and F1 of 0.84. TF-IDF has perfect spam precision (no ham flagged as spam), but it misses many spam messages (44 vs 16). TF-IDF also lowers the feature values, which makes `MultinomialNB` lean more towards the majority class (ham) on this imbalanced dataset. Because `MultinomialNB` is built for count data, and because catching spam matters, **CountVectorizer with stop words is the better feature for `spam.csv`**.